In [ ]:
# DFU Phase-4 TRUE COLAB ONLY V5 — Drive-only, CPU/GPU auto, resume-safe
import urllib.request, hashlib
from pathlib import Path

VERSION = "DFU_PHASE4_TRUE_COLAB_ONLY_V5_20260812"
SOURCE_COMMIT = "518c25a4dc3f103fad4a8e0545c3f6dd729c0434"
PARTS = [
    ("scripts/phase4_universal_v2_parts/part_01.pyfrag", "1280e699f2c02e74097cff69dd52db37cea77251"),
    ("scripts/phase4_universal_v2_parts/part_02.pyfrag", "ea776ea1476a6cbe41913f116887d605310c985f"),
    ("scripts/phase4_universal_v2_parts/part_03.pyfrag", "915202b01279b434fcbca6955c48b10bae13c942"),
    ("scripts/phase4_universal_v2_parts/part_04.pyfrag", "5e5128456b99877957dcecf341d3c149137acfbc"),
    ("scripts/phase4_universal_v2_parts/part_05.pyfrag", "db01f06aaa6519fef856903c20ba0f939ef5e14d"),
]
BASE = f"https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/{SOURCE_COMMIT}/"

def git_blob_sha(raw):
    return hashlib.sha1(b"blob " + str(len(raw)).encode() + b"\0" + raw).hexdigest()

print("=" * 100)
print(VERSION)
print("TRUE GOOGLE COLAB ONLY | GOOGLE DRIVE | CPU/GPU AUTO | RESUME-SAFE")
print("NO TRAINING | NO FINE-TUNING | NO EXTERNAL THRESHOLD/CALIBRATION FITTING")
print("=" * 100)

if Path("/kaggle").exists():
    raise RuntimeError("KAGGLE RUNTIME DETECTED. Open this notebook in a fresh Google Colab tab.")

try:
    import google.colab
except Exception as e:
    raise RuntimeError("GOOGLE COLAB RUNTIME NOT DETECTED. Use a Google-hosted Colab runtime.") from e

chunks = []
for path, expected in PARTS:
    raw = urllib.request.urlopen(BASE + path, timeout=120).read()
    actual = git_blob_sha(raw)
    if actual != expected:
        raise RuntimeError(f"Source fragment mismatch for {path}: expected={expected} actual={actual}")
    print("Source fragment PASS:", path, actual)
    chunks.append(raw)

source = b"".join(chunks)
print("Original combined source SHA256:", hashlib.sha256(source).hexdigest())
text = source.decode("utf-8")
compile(text, "dfu_phase4_original.py", "exec")
print("Original combined source compile: PASS")

old_env = '    env = detect_environment()\n'
new_env = (
    '    if Path("/kaggle").exists():\n'
    '        raise RuntimeError("Kaggle runtime detected inside Colab-only runner")\n'
    '    try:\n'
    '        import google.colab\n'
    '    except Exception as e:\n'
    '        raise RuntimeError("Google Colab runtime not detected") from e\n'
    '    env = "colab"\n'
)
if text.count(old_env) != 1:
    raise RuntimeError(f"Environment patch target count={text.count(old_env)}, expected 1")
text = text.replace(old_env, new_env)

old_guard = (
    '    import torch\n'
    '    if not torch.cuda.is_available():\n'
    '        raise RuntimeError(\n'
    '            "CUDA GPU not available. CPU overlap audit has been cached, but frozen model inference is intentionally blocked. "\n'
    '            "Enable a Kaggle/Colab GPU accelerator and rerun; the audit will be RESTORED."\n'
    '        )\n'
    '    device = torch.device("cuda")\n'
    '    print("Inference device: cuda", flush=True)\n'
    '    print("GPU:", torch.cuda.get_device_name(0), flush=True)\n'
)
new_guard = (
    '    import torch\n'
    '    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n'
    '    print("Inference device:", device, flush=True)\n'
    '    if device.type == "cuda":\n'
    '        print("GPU:", torch.cuda.get_device_name(0), flush=True)\n'
    '    else:\n'
    '        print("CPU inference enabled. This is slower but valid and resume-safe.", flush=True)\n'
)
if text.count(old_guard) != 1:
    raise RuntimeError(f"CPU/GPU patch target count={text.count(old_guard)}, expected 1")
text = text.replace(old_guard, new_guard)

replacements = [
    ('print("Overlap audit is CPU/I/O by design and is cached. Model inference is GPU-required.")',
     'print("Overlap audit is CPU/I/O by design and is cached. Model inference uses GPU if available, otherwise CPU.")'),
    ('print(f"GPU INFERENCE START: {model_key} s{seed} f{fold}", flush=True)',
     'print(f"INFERENCE START ({device.type}): {model_key} s{seed} f{fold}", flush=True)'),
    ('print(f"GPU INFERENCE PASS: {model_key} s{seed} f{fold}", flush=True)',
     'print(f"INFERENCE PASS ({device.type}): {model_key} s{seed} f{fold}", flush=True)'),
    ('"gpu_name": torch.cuda.get_device_name(0),',
     '"gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'),
    ('print("Inference device: cuda")\n    print("GPU:", torch.cuda.get_device_name(0))',
     'print("Inference device:", device)\n    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - CPU RUN")'),
    ('VERSION = "DFU_PHASE4_UNIVERSAL_KAGGLE_COLAB_V2_20260812"',
     'VERSION = "DFU_PHASE4_TRUE_COLAB_ONLY_V5_20260812"'),
]
for old, new in replacements:
    c = text.count(old)
    if c != 1:
        raise RuntimeError(f"Patch target count={c}, expected 1 for: {old[:80]}")
    text = text.replace(old, new)

compile(text, "dfu_phase4_true_colab_v5.py", "exec")
print("Patched source compile: PASS")
print("Patched source SHA256:", hashlib.sha256(text.encode()).hexdigest())
print("Starting TRUE Colab Phase-4...")
exec(compile(text, "dfu_phase4_true_colab_v5.py", "exec"), globals())
